In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/AmesHousing.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'AmesHousing.csv'

In [ ]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

In [ ]:
null_col_df.dtypes

In [ ]:
values = {
    # "Lot Frontage": imputation,
    "Alley": "No Alley Access",
    "Mas Vnr Type": "None",
    # "Mas Vnr Area": imputation,
    "Bsmt Qual": "No Basement",
    "Bsmt Cond": "No Basement",
    "Bsmt Exposure" : "No Basement",
    "BsmtFin Type 1" : "No Basement",
    # "BsmtFin SF 1" : imputation,
    "BsmtFin Type 2" : "No Basement",
    # "BsmtFin SF 2" : imputation,
    # "Bsmt Unf SF" : imputation,
    # "Total Bsmt SF" : imputation,
    # "Electrical" : might actually be missing,
    # "Bsmt Full Bath" : imputation,
    # "Bsmt Half Bath" : imputation,
    "Fireplace Qu" : "No Fireplace",
    "Garage Type" : "No Garage",
    # "Garage Yr Blt" : might actually be missing,
    "Garage Finish" : "No Garage",
    # "Garage Cars" : imputation,
    # "Garage Area" : imputation,
    "Garage Qual" : "No Garage",
    "Garage Cond" : "No Garage",
    "Pool QC" : "No Pool",
    "Fence" : "No Fence",
    "Misc Feature" : "None"
}

In [ ]:
inconsistency1 = (df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] > 0) # if Type is NA AND Area > 0
inconsistency2 = (~df['Mas Vnr Type'].isna()) & (df['Mas Vnr Area'] == 0) # if Type is not NA AND Area == 0

In [ ]:
df[df['Mas Vnr Area'] < 0] # area should be nonnegative

In [ ]:
df[inconsistency1][['Mas Vnr Type', 'Mas Vnr Area']]

In [ ]:
df[inconsistency2][['Mas Vnr Type', 'Mas Vnr Area']]

In [ ]:
df.loc[inconsistency1, 'Mas Vnr Type'] = 'Unknown'
df.loc[inconsistency2, 'Mas Vnr Type'] = 'None'

In [ ]:
bsmt_cat_var = ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']
bsmt_num_var = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath']
bsmt_vars = bsmt_cat_var + bsmt_num_var
bsmt_row_nas = df[bsmt_cat_var].isna().sum(axis=1)
inconsistency3 = (bsmt_row_nas < len(bsmt_cat_var)) & (bsmt_row_nas > 0) # the bsmt categorical features contradict each other
inconsistency4 = (df[bsmt_cat_var].isna().sum(axis=1) == len(bsmt_cat_var)) & (df[bsmt_num_var].sum(axis=1) > 0) # bsmt numerical features contradict bsmt categorical features

In [ ]:
df[df[bsmt_num_var].sum(axis=1) < 0] # all numeric features should be nonnegative

In [ ]:
df[inconsistency3][bsmt_vars]

In [ ]:
df[inconsistency4][bsmt_vars]

In [ ]:
df.loc[inconsistency3, bsmt_cat_var] = df.loc[inconsistency3, bsmt_cat_var].fillna('Unknown')

In [ ]:
garage_cat_var = ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']
garage_num_var = ['Garage Yr Blt', 'Garage Cars', 'Garage Area']
garage_vars = garage_cat_var + garage_num_var
garage_row_nas = df[garage_cat_var].isna().sum(axis=1)
inconsistency5 = (garage_row_nas < len(garage_cat_var)) & (garage_row_nas > 0) # the garage categorical features contradict each other
inconsistency6 = (df[garage_cat_var].isna().sum(axis=1) == len(garage_cat_var)) & (df[garage_num_var].sum(axis=1) > 0) # garage numerical features contradict garage categorical features

In [ ]:
df[df[garage_num_var].sum(axis=1) < 0] # all numeric columns should be nonnegative

In [ ]:
df[inconsistency5][garage_vars]

In [ ]:
df.loc[inconsistency5, garage_cat_var] = df.loc[inconsistency5, garage_cat_var].fillna('Unknown')

In [ ]:
df[df['Garage Type'] == 'Detchd'][garage_num_var].median()

In [ ]:
df.loc[inconsistency5, 'Garage Yr Blt'] = 1962
df.loc[inconsistency5, 'Garage Cars'] = df.loc[inconsistency5, 'Garage Cars'].fillna(2)
df.loc[inconsistency5, 'Garage Area'] = df.loc[inconsistency5, 'Garage Area'].fillna(400)

In [ ]:
df[inconsistency6][garage_vars]

In [ ]:
df[(df['Fireplaces'] > 0) & (df['Fireplace Qu'].isna())]

In [ ]:
df[(df['Pool Area'] > 0) & (df['Pool QC'].isna())]

In [ ]:
df[(df['Misc Val'] > 0) & (df['Misc Feature'].isna())]

In [ ]:
bsmt_total_check = df['BsmtFin SF 1'] + df['BsmtFin SF 2'] + df['Bsmt Unf SF'] == df['Total Bsmt SF']

In [ ]:
df[bsmt_total_check][bsmt_vars]

In [ ]:
df[~bsmt_total_check][bsmt_vars]

In [ ]:
live_area_check = df['1st Flr SF'] + df['2nd Flr SF'] + df['Low Qual Fin SF'] == df['Gr Liv Area']

In [ ]:
df[live_area_check][['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area']]

In [ ]:
garage_consist_check1 = (df['Garage Cars'] > 0) & (df['Garage Area'] > 0)
garage_consist_check2 = (df['Garage Cars'] == 0) & (df['Garage Area'] == 0)
df[~garage_consist_check1][['Garage Cars', 'Garage Area']]

In [ ]:
df[garage_consist_check2][['Garage Cars', 'Garage Area']]

In [ ]:
(df[~garage_consist_check1][['Garage Cars', 'Garage Area']] != df[garage_consist_check2][['Garage Cars', 'Garage Area']]).sum()

In [ ]:
temp = df.dtypes.reset_index()
print(temp[0].value_counts())
temp = temp[temp[0] != 'object']
numeric_cols = temp['index'].unique()
categorical_cols = [col for col in df.columns if col not in numeric_cols]
df[df[numeric_cols].sum(axis=1) < 0]

In [ ]:
year_consist_check1 = (df['Year Remod/Add'] > df['Year Built']) | (df['Year Remod/Add'] == df['Year Built'])
index1 = df[~year_consist_check1].index
df[~year_consist_check1][['Year Remod/Add', 'Year Built']]

In [ ]:
year_consist_check2 = df['Yr Sold'] >= df['Year Built']
index2 = df[~year_consist_check2].index
df[~year_consist_check2][['Yr Sold', 'Year Built']]

In [ ]:
year_consist_check3 = df['Yr Sold'] >= df['Garage Yr Blt']
index3 = df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()].index
df[~year_consist_check3 & ~df['Garage Yr Blt'].isna()][['Yr Sold', 'Garage Yr Blt']]

In [ ]:
df.loc[index1,['Year Remod/Add', 'Year Built']] = np.nan
df.loc[index2, ['Yr Sold', 'Year Built']] = np.nan
df.loc[index3, ['Yr Sold', 'Garage Yr Blt']] = np.nan

In [ ]:
df.loc[list(index1) + list(index2) + list(index3), ['Year Remod/Add', 'Yr Sold', 'Year Built', 'Garage Yr Blt']]

In [ ]:
index = df[df['Garage Yr Blt'].isna()].index
df.loc[df['Year Built'].isna(), 'Year Built'] = df['Year Built'].median()
df.loc[index, 'Garage Yr Blt'] = df.loc[index, 'Year Built']
df.loc[df['Year Remod/Add'].isna(), 'Year Remod/Add'] = df['Year Remod/Add'].median()
df.loc[df['Yr Sold'].isna(), 'Yr Sold'] = df['Yr Sold'].median()

In [ ]:
df.fillna(value=values, inplace=True)

In [ ]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

In [ ]:
df.loc[df['BsmtFin SF 1'].isna(), 'BsmtFin SF 1'] = df['BsmtFin SF 1'].median()
df.loc[df['BsmtFin SF 2'].isna(), 'BsmtFin SF 2'] = df['BsmtFin SF 2'].median()
df.loc[df['Bsmt Unf SF'].isna(), 'Bsmt Unf SF'] = df['Bsmt Unf SF'].median()
index = df.loc[df['Total Bsmt SF'].isna(), 'Total Bsmt SF'].index
df.loc[index, 'Total Bsmt SF'] = (df['BsmtFin SF 1'] + df['BsmtFin SF 2'] + df['Bsmt Unf SF']).loc[index]

In [ ]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

In [ ]:
# group-based imputation
df['Lot Frontage'] = df['Lot Frontage'].fillna(
    df.groupby(['Neighborhood', 'Lot Config'])['Lot Frontage'].transform('median')
)
# global imputation for the 7 rows still missing
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

In [ ]:
df[df['Mas Vnr Area'].isna()][['Mas Vnr Type', 'Mas Vnr Area']]

In [ ]:
df.loc[df['Mas Vnr Area'].isna(), 'Mas Vnr Area'] = 0

In [ ]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

In [ ]:
df['Electrical'].mode()[0]

In [ ]:
df.loc[df['Electrical'].isna(), 'Electrical'] = df['Electrical'].mode()[0]
df.loc[df['Bsmt Full Bath'].isna(), 'Bsmt Full Bath'] = df['Bsmt Full Bath'].mode()[0]
df.loc[df['Bsmt Half Bath'].isna(), 'Bsmt Half Bath'] = df['Bsmt Half Bath'].mode()[0]

In [ ]:
mask = df.isna().any()
null_col_df = df.loc[:, mask]
null_col_df.isna().sum()

In [ ]:
df.to_csv("data/ames_cleaned.csv", index=False)